# CampaignOptima: Phase 4 — Model Explainability (XAI) with SHAP

**Objective:** Ensure complete regulatory transparency and marketing interpretability:
1. **Global Interpretability:** Summary and feature importance plots showing what drives term deposit conversion across the broader population.
2. **Local Interpretability:** Customer-level force and waterfall explanations showing why specific individuals are flagged as high vs low propensity.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data_loader import BankMarketingDataLoader
from src.features import CampaignOptimaFeaturePipeline
from src.explainability import ModelExplainabilityEngine
from xgboost import XGBClassifier

## 1. Train Model & Initialize SHAP TreeExplainer

In [ ]:
loader = BankMarketingDataLoader()
df_raw = loader.load_raw_data()
X, y = loader.clean_and_prepare(df_raw)
X_train, X_test, y_train, y_test = loader.get_stratified_train_test_split(X, y)

pipeline = CampaignOptimaFeaturePipeline()
X_train_proc = pipeline.fit_transform(X_train)
X_test_proc = pipeline.transform(X_test)

model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42)
model.fit(X_train_proc, y_train)

xai_engine = ModelExplainabilityEngine(model, feature_names=pipeline.feature_names_out_)
print("SHAP Explainer initialized successfully.")

## 2. Global Interpretability: Mean Absolute SHAP Importance

In [ ]:
X_sample = X_test_proc.sample(200, random_state=42)
global_imp = xai_engine.get_global_feature_importance(X_sample, top_n=12)

plt.figure(figsize=(9, 5))
plt.barh(global_imp["feature"][::-1], global_imp["mean_abs_shap"][::-1], color="#2563EB")
plt.title("Global Feature Importance: Mean |SHAP Value|")
plt.xlabel("Mean Absolute Impact on Propensity Log-Odds")
plt.tight_layout()
plt.show()

## 3. Global SHAP Summary Plot

In [ ]:
shap_values = xai_engine.calculate_shap_values(X_sample)
shap.summary_plot(shap_values, X_sample, feature_names=pipeline.feature_names_out_, max_display=12)

## 4. Local Interpretability: High-Propensity Customer Attribution

In [ ]:
# Find high propensity customer
probs = model.predict_proba(X_sample)[:, 1]
high_prop_idx = np.argmax(probs)
customer_record = X_sample.iloc[[high_prop_idx]]

print(f"High Propensity Customer Conversion Probability: {probs[high_prop_idx]:.2%}")
explanation = xai_engine.explain_individual_customer(customer_record, top_k=8)
pd.DataFrame(explanation["top_drivers"])